# 🏭 CASO 3 — PRODUCCIÓN INDUSTRIAL
## Visualización de Datos con Streamlit
### ✅ TAREA RESUELTA

**Empresa:** MetalParts Colombia S.A.S. — Manufactura de piezas metálicas (Itagüí, Antioquia)

Este notebook contiene el análisis exploratorio del dataset de producción y prepara el archivo `caso3_produccion_app.py` con el dashboard Streamlit.

**Preguntas a responder:**
1. ¿Cuál es la eficiencia promedio por línea de producción?
2. ¿En qué turno se producen más defectos?
3. ¿Qué máquina genera más tiempos de paro?
4. ¿Cómo ha evolucionado la producción semana a semana?
5. ¿Cuál es la relación entre temperatura y tasa de defectos?


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 1 — Importar librerías
# ══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.float_format', '{:,.2f}'.format)
print('✅ Librerías cargadas')


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 2 — Cargar y explorar el dataset
# ══════════════════════════════════════════════════════════════
df = pd.read_csv('caso3_produccion_dataset.csv')
df['fecha_produccion'] = pd.to_datetime(df['fecha_produccion'])

print('Shape:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nPrimeras filas:')
df.head()


In [ ]:
# Estadísticas descriptivas y valores únicos
print('Valores nulos:', df.isnull().sum().sum())
print('Líneas:', df['linea_produccion'].unique())
print('Turnos:', df['turno'].unique())
print('Máquinas:', df['maquina'].unique())
print('Causas de paro:', df['causa_paro'].unique())
print('Rango fechas:', df['fecha_produccion'].min().date(), 'a', df['fecha_produccion'].max().date())
print('\nDescribe numérico:')
df.describe().round(2)


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 3 — KPIs del negocio
# ══════════════════════════════════════════════════════════════
eficiencia_prom   = df['eficiencia_pct'].mean()
defectos_prom     = df['tasa_defectos_pct'].mean()
total_producido   = df['unidades_producidas'].sum()
total_defectuosas = df['unidades_defectuosas'].sum()
costo_total       = df['costo_produccion_cop'].sum()
paro_total_min    = df['tiempo_paro_min'].sum()
energia_total     = df['consumo_energia_kwh'].sum()

print(f'📊 Eficiencia promedio:     {eficiencia_prom:6.2f} %')
print(f'⚠️  Tasa de defectos prom.: {defectos_prom:6.2f} %')
print(f'📦 Total unidades:          {total_producido:>10,.0f}')
print(f'❌ Unidades defectuosas:    {total_defectuosas:>10,.0f}')
print(f'💰 Costo total producción:  ${costo_total:>13,.0f} COP')
print(f'⏱️  Tiempo de paro total:   {paro_total_min:>10,.0f} min  ({paro_total_min/60:,.1f} h)')
print(f'⚡ Energía total consumida: {energia_total:>10,.0f} kWh')


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 4 — Análisis por grupos
# ══════════════════════════════════════════════════════════════

# Eficiencia por línea
ef_linea = df.groupby('linea_produccion')['eficiencia_pct'].mean().round(2).sort_values(ascending=False)
print('🏭 Eficiencia promedio por línea:')
print(ef_linea, '\n')

# Defectos por turno
def_turno = df.groupby('turno')['tasa_defectos_pct'].mean().round(2).sort_values(ascending=False)
print('🌙 Tasa de defectos promedio por turno:')
print(def_turno, '\n')

# Paro por máquina
paro_maq = df.groupby('maquina')['tiempo_paro_min'].sum().round(0).sort_values(ascending=False)
print('🛑 Tiempo de paro total por máquina (min):')
print(paro_maq, '\n')

# Producción semanal
prod_sem = df.groupby('semana').agg(
    unidades=('unidades_producidas', 'sum'),
    eficiencia=('eficiencia_pct', 'mean'),
    defectos=('tasa_defectos_pct', 'mean')
).round(2)
print('📅 Producción semanal (resumen):')
prod_sem.head()


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 5 — Visualización 1: Eficiencia por línea (Boxplot)
# ══════════════════════════════════════════════════════════════
fig1 = px.box(
    df,
    x='linea_produccion', y='eficiencia_pct',
    color='linea_produccion',
    title='<b>Eficiencia (%) por Línea de Producción</b>',
    labels={'linea_produccion': 'Línea', 'eficiencia_pct': 'Eficiencia (%)'},
    points='all'
)
fig1.update_layout(showlegend=False, height=450)
fig1.add_hline(y=df['eficiencia_pct'].mean(), line_dash='dash',
               annotation_text=f'Promedio general: {df["eficiencia_pct"].mean():.1f}%')
fig1.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 6 — Visualización 2: Defectos por turno y línea
# ══════════════════════════════════════════════════════════════
defectos_turno_linea = df.groupby(['turno', 'linea_produccion'])['tasa_defectos_pct'].mean().reset_index()

fig2 = px.bar(
    defectos_turno_linea,
    x='turno', y='tasa_defectos_pct', color='linea_produccion',
    barmode='group',
    title='<b>Tasa de defectos promedio (%) por Turno y Línea</b>',
    labels={'turno': 'Turno', 'tasa_defectos_pct': 'Tasa de defectos (%)',
            'linea_produccion': 'Línea'},
    text_auto='.2f'
)
fig2.update_layout(height=450)
fig2.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 7 — Visualización 3: Tiempo de paro por máquina
# ══════════════════════════════════════════════════════════════
paro_df = paro_maq.reset_index()
paro_df.columns = ['maquina', 'paro_total_min']

fig3 = px.bar(
    paro_df.sort_values('paro_total_min'),
    x='paro_total_min', y='maquina',
    orientation='h',
    color='paro_total_min',
    color_continuous_scale='Reds',
    title='<b>Tiempo de paro total por Máquina (minutos)</b>',
    labels={'maquina': 'Máquina', 'paro_total_min': 'Paro total (min)'},
    text='paro_total_min'
)
fig3.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig3.update_layout(height=450, coloraxis_showscale=False)
fig3.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 8 — Visualización 4: Evolución semanal de la producción
# ══════════════════════════════════════════════════════════════
prod_sem_reset = prod_sem.reset_index()

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=prod_sem_reset['semana'], y=prod_sem_reset['unidades'],
    mode='lines+markers', name='Unidades producidas',
    line=dict(color='#1f77b4', width=2.5)
))
fig4.add_trace(go.Scatter(
    x=prod_sem_reset['semana'], y=prod_sem_reset['eficiencia']*100,
    mode='lines', name='Eficiencia x100 (referencia)',
    line=dict(color='#ff7f0e', dash='dot'), yaxis='y2', visible='legendonly'
))
fig4.update_layout(
    title='<b>Producción semanal — Año 2024</b>',
    xaxis_title='Semana del año', yaxis_title='Unidades producidas',
    height=450
)
fig4.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 9 — Visualización 5: Temperatura vs Tasa de defectos
# ══════════════════════════════════════════════════════════════
fig5 = px.scatter(
    df,
    x='temperatura_c', y='tasa_defectos_pct',
    color='linea_produccion', size='unidades_producidas',
    trendline='ols', trendline_scope='overall',
    title='<b>Temperatura del proceso vs Tasa de defectos</b>',
    labels={'temperatura_c': 'Temperatura (°C)',
            'tasa_defectos_pct': 'Tasa de defectos (%)',
            'linea_produccion': 'Línea'},
    hover_data=['maquina', 'producto']
)
fig5.update_layout(height=500)
fig5.show()

# Correlación
corr = df['temperatura_c'].corr(df['tasa_defectos_pct'])
print(f'Coeficiente de correlación de Pearson (temperatura vs defectos): {corr:.3f}')


## 💡 Insights y hallazgos

### 1. La Línea D está rezagada
La **Línea D tiene una eficiencia promedio del 72.15%**, frente al **80.20% de la Línea C** (la mejor). Son **~8 puntos porcentuales de diferencia**, suficiente para representar miles de unidades perdidas al año. La Línea D debe ser auditada con prioridad.

### 2. El turno Mañana, contraintuitivamente, es el más problemático
La gente suele asumir que el turno Noche acumula más defectos por fatiga, pero los datos muestran lo contrario: **Mañana → 6.18% de defectos, Noche → 4.82%**. Posibles hipótesis: arranque en frío de máquinas, calibración inicial deficiente, alto volumen de producción al inicio del día.

### 3. Torno-02 es el cuello de botella en mantenimiento
**Torno-02 acumula 2,116 minutos de paro (≈ 35 horas)**, casi el doble que Prensa-01 o CNC-02. Es la máquina con mayor riesgo operativo y debería entrar a un plan de mantenimiento preventivo intensivo.

### 4. La temperatura NO es la causa principal de defectos
La correlación de Pearson entre temperatura y tasa de defectos es de **~0.12 (muy débil)**. La temperatura del proceso explica muy poco de la variación en defectos; el problema está en otro lado (turno, línea, máquina, operador).

### 📌 Recomendación para la gerencia
1. **Priorizar la Línea D**: auditar máquinas, operadores y procedimientos de esa línea.
2. **Revisar el protocolo de arranque del turno Mañana** (calibración, precalentamiento).
3. **Plan de mantenimiento preventivo intensivo para Torno-02**, incluyendo análisis de causa raíz de las fallas eléctricas.
4. **No invertir en control de temperatura** como medida principal contra defectos: la evidencia no lo justifica.

---


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 10 — App Streamlit
# Genera el archivo caso3_produccion_app.py con el dashboard completo.
# Para ejecutar:  streamlit run caso3_produccion_app.py
# ══════════════════════════════════════════════════════════════

app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ─── Configuración de página ──────────────────────────────────
st.set_page_config(
    page_title="MetalParts — Dashboard de Producción",
    page_icon="🏭",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─── Carga de datos (cacheada) ─────────────────────────────────
@st.cache_data
def cargar_datos(ruta="caso3_produccion_dataset.csv"):
    df = pd.read_csv(ruta)
    df["fecha_produccion"] = pd.to_datetime(df["fecha_produccion"])
    return df

df = cargar_datos()

# ─── Sidebar: filtros ─────────────────────────────────────────
st.sidebar.title("🎛️ Filtros")

lineas = st.sidebar.multiselect(
    "Línea de producción",
    options=sorted(df["linea_produccion"].unique()),
    default=sorted(df["linea_produccion"].unique()),
)

turnos = st.sidebar.multiselect(
    "Turno",
    options=df["turno"].unique().tolist(),
    default=df["turno"].unique().tolist(),
)

maquinas = st.sidebar.multiselect(
    "Máquina",
    options=sorted(df["maquina"].unique()),
    default=sorted(df["maquina"].unique()),
)

# Bonus: rango de fechas
fecha_min, fecha_max = df["fecha_produccion"].min(), df["fecha_produccion"].max()
rango_fechas = st.sidebar.date_input(
    "Rango de fechas",
    value=(fecha_min, fecha_max),
    min_value=fecha_min, max_value=fecha_max,
)

# Aplicar filtros
df_f = df[
    df["linea_produccion"].isin(lineas)
    & df["turno"].isin(turnos)
    & df["maquina"].isin(maquinas)
]
if len(rango_fechas) == 2:
    df_f = df_f[
        (df_f["fecha_produccion"] >= pd.to_datetime(rango_fechas[0]))
        & (df_f["fecha_produccion"] <= pd.to_datetime(rango_fechas[1]))
    ]

st.sidebar.markdown("---")
st.sidebar.metric("Órdenes filtradas", f"{len(df_f):,}")
st.sidebar.download_button(
    "⬇️ Descargar datos filtrados (CSV)",
    data=df_f.to_csv(index=False).encode("utf-8"),
    file_name="produccion_filtrada.csv",
    mime="text/csv",
)

# ─── Encabezado ───────────────────────────────────────────────
st.title("🏭 MetalParts Colombia — Dashboard de Producción")
st.caption("Zona Industrial de Itagüí · Datos 2024")

if df_f.empty:
    st.warning("No hay datos con los filtros seleccionados.")
    st.stop()

# ─── KPIs (patrón Z) ───────────────────────────────────────────
k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("📊 Eficiencia prom.", f"{df_f['eficiencia_pct'].mean():.1f}%")
k2.metric("⚠️ Defectos prom.",  f"{df_f['tasa_defectos_pct'].mean():.2f}%")
k3.metric("📦 Unidades prod.",  f"{df_f['unidades_producidas'].sum():,}")
k4.metric("💰 Costo total",     f"${df_f['costo_produccion_cop'].sum()/1e6:,.1f}M")
k5.metric("⏱️ Paro total",      f"{df_f['tiempo_paro_min'].sum():,.0f} min")

st.markdown("---")

# ─── Tabs con las gráficas ────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["📈 Operación", "🛑 Calidad y Paros", "🚨 Alertas"])

with tab1:
    c1, c2 = st.columns(2)
    with c1:
        fig = px.box(
            df_f, x="linea_produccion", y="eficiencia_pct",
            color="linea_produccion", points="all",
            title="Eficiencia (%) por Línea",
            labels={"linea_produccion": "Línea", "eficiencia_pct": "Eficiencia (%)"},
        )
        fig.update_layout(showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        prod_sem = df_f.groupby("semana")["unidades_producidas"].sum().reset_index()
        fig = px.line(
            prod_sem, x="semana", y="unidades_producidas", markers=True,
            title="Producción semanal",
            labels={"semana": "Semana", "unidades_producidas": "Unidades"},
        )
        st.plotly_chart(fig, use_container_width=True)

with tab2:
    c1, c2 = st.columns(2)
    with c1:
        defectos = df_f.groupby(["turno", "linea_produccion"])["tasa_defectos_pct"].mean().reset_index()
        fig = px.bar(
            defectos, x="turno", y="tasa_defectos_pct", color="linea_produccion",
            barmode="group", text_auto=".2f",
            title="Tasa de defectos por Turno y Línea",
            labels={"turno": "Turno", "tasa_defectos_pct": "Defectos (%)",
                    "linea_produccion": "Línea"},
        )
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        paro = df_f.groupby("maquina")["tiempo_paro_min"].sum().sort_values().reset_index()
        fig = px.bar(
            paro, x="tiempo_paro_min", y="maquina", orientation="h",
            color="tiempo_paro_min", color_continuous_scale="Reds",
            title="Paro total por Máquina (min)",
            labels={"maquina": "Máquina", "tiempo_paro_min": "Paro (min)"},
        )
        fig.update_layout(coloraxis_showscale=False)
        st.plotly_chart(fig, use_container_width=True)

    st.markdown("#### Temperatura vs Defectos")
    fig = px.scatter(
        df_f, x="temperatura_c", y="tasa_defectos_pct",
        color="linea_produccion", size="unidades_producidas",
        trendline="ols", trendline_scope="overall",
        hover_data=["maquina", "producto"],
        labels={"temperatura_c": "Temperatura (°C)",
                "tasa_defectos_pct": "Defectos (%)",
                "linea_produccion": "Línea"},
    )
    st.plotly_chart(fig, use_container_width=True)

with tab3:
    st.subheader("🚨 Órdenes con tasa de defectos > 10%")
    alertas = df_f[df_f["tasa_defectos_pct"] > 10].sort_values(
        "tasa_defectos_pct", ascending=False
    )
    if alertas.empty:
        st.success("✅ Ninguna orden supera el 10% de defectos en el filtro actual.")
    else:
        st.warning(f"Se encontraron {len(alertas)} órdenes que requieren revisión.")
        st.dataframe(
            alertas[[
                "id_orden", "fecha_produccion", "linea_produccion", "turno",
                "maquina", "producto", "unidades_producidas",
                "unidades_defectuosas", "tasa_defectos_pct", "causa_paro",
            ]],
            use_container_width=True, hide_index=True,
        )

st.markdown("---")
st.caption("Dashboard construido con Streamlit + Plotly · Caso 3 — Producción Industrial")
'''

with open('caso3_produccion_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('✅ Archivo caso3_produccion_app.py creado')
print('▶️  Ejecuta:  streamlit run caso3_produccion_app.py')
